In [ ]:
import os
import re
import pandas as pd
import subprocess
from tqdm import tqdm
import csv
import shutil

# Function to slow down a video
def slow_down_video(input_path, output_path, speed_factor=0.5):
    """
    Slow down a video by the specified factor using ffmpeg.
    
    Args:
        input_path: Path to the input video
        output_path: Path to save the output video
        speed_factor: Factor to slow down the video (e.g., 0.5 for half speed)
    """
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Build ffmpeg command: use setpts filter to slow down video, atempo to slow down audio
    # atempo filter only supports 0.5-2.0 range, so for extreme slowing, we chain it
    cmd = [
        'ffmpeg', 
        '-i', input_path, 
        '-filter_complex', 
        f'[0:v]setpts={1/speed_factor}*PTS[v];[0:a]atempo={speed_factor}[a]', 
        '-map', '[v]', 
        '-map', '[a]', 
        '-c:v', 'libx264', 
        '-preset', 'medium', 
        '-c:a', 'aac', 
        output_path,
        '-y'  # Overwrite output file if it exists
    ]
    
    subprocess.run(cmd, check=True)

# Function to adjust SRT timestamps
def adjust_srt_timestamps(srt_text, speed_factor=0.5):
    """
    Adjust SRT timestamps by the specified speed factor.
    
    Args:
        srt_text: SRT format text
        speed_factor: Factor to slow down the timestamps (e.g., 0.5 for double duration)
    
    Returns:
        Adjusted SRT text
    """
    # Define a regex pattern to match SRT timestamp lines
    pattern = r'(\d+:\d+:\d+,\d+) --> (\d+:\d+:\d+,\d+)'
    
    def time_to_seconds(time_str):
        """Convert SRT timestamp to seconds"""
        h, m, s = time_str.replace(',', '.').split(':')
        return float(h) * 3600 + float(m) * 60 + float(s)
    
    def seconds_to_time(seconds):
        """Convert seconds to SRT timestamp format"""
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        s = seconds % 60
        ms = int((s - int(s)) * 1000)
        s = int(s)
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"
    
    def adjust_timestamp(match):
        """Adjust a pair of timestamps based on the speed factor"""
        start_time = match.group(1)
        end_time = match.group(2)
        
        # Convert to seconds
        start_seconds = time_to_seconds(start_time)
        end_seconds = time_to_seconds(end_time)
        
        # Adjust time by dividing by speed factor (e.g., 0.5 makes duration 2x longer)
        adjusted_start = start_seconds / speed_factor
        adjusted_end = end_seconds / speed_factor
        
        # Convert back to timestamp format
        adjusted_start_time = seconds_to_time(adjusted_start)
        adjusted_end_time = seconds_to_time(adjusted_end)
        
        return f"{adjusted_start_time} --> {adjusted_end_time}"
    
    # Use regex to replace all timestamp lines
    adjusted_srt = re.sub(pattern, adjust_timestamp, srt_text)
    return adjusted_srt

# Main execution

# Create output folder
output_folder = "videos_slow"
os.makedirs(output_folder, exist_ok=True)

# Process metadata file
metadata_file = "video_metadata.csv"
output_metadata_file = "video_slow_metadata.csv"

print("Reading metadata file...")
df = pd.read_csv(metadata_file)

# Create copies of dataframe columns to modify
df['slowed_video_path'] = df['video_path'].copy()
df['slowed_caption'] = df['caption'].copy()

# Process each video and update metadata
print("Processing videos and updating metadata...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    # Get video info
    video_path = row['video_path']
    
    # Check if video exists
    if not os.path.exists(video_path):
        print(f"Warning: Video not found: {video_path}")
        continue
    
    # Create output path
    output_path = video_path.replace("videos/", "videos_slow/")
    
    # Slow down the video
    try:
        slow_down_video(video_path, output_path)
        
        # Update metadata
        df.at[idx, 'slowed_video_path'] = output_path
        
        # Update caption timestamps if available
        if pd.notna(row['caption']):
            adjusted_caption = adjust_srt_timestamps(row['caption'])
            df.at[idx, 'slowed_caption'] = adjusted_caption
    
    except Exception as e:
        print(f"Error processing {video_path}: {str(e)}")

# Create final output dataframe
output_df = df.copy()
output_df['video_path'] = df['slowed_video_path']  # Replace with slowed path
output_df['caption'] = df['slowed_caption']  # Replace with adjusted captions
output_df = output_df.drop(['slowed_video_path', 'slowed_caption'], axis=1)  # Remove temporary columns
# Double the duration of the video
output_df['duration'] = output_df['duration'] * 2

# Save updated metadata to CSV
print(f"Saving updated metadata to {output_metadata_file}...")
output_df.to_csv(output_metadata_file, index=False)

print("Processing complete!")

# Show a summary of the processing
num_videos = len(df)
num_processed = sum([1 for idx, row in df.iterrows() if os.path.exists(row['slowed_video_path'])])
print(f"Summary: Processed {num_processed} out of {num_videos} videos")
print(f"Slow videos saved to: {output_folder}")
print(f"Updated metadata saved to: {output_metadata_file}")

Reading metadata file...
Processing videos and updating metadata...


  0%|          | 0/281 [00:00<?, ?it/s]

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Saving updated metadata to video_slow_metadata.csv...
Processing complete!
Summary: Processed 281 out of 281 videos
Slow videos saved to: videos_slow
Updated metadata saved to: video_slow_metadata.csv
